In [31]:
import ee
import os
import geemap
from dotenv import load_dotenv
load_dotenv()

# Initialize the Earth Engine module.
PROJECT_ID = os.getenv("PROJECT_ID")
ee.Initialize(project = os.getenv("PROJECT_ID"))

# Define Denmark Area
DENMARK = ee.FeatureCollection('USDOS/LSIB_SIMPLE/2017').filter(ee.Filter.eq('country_na', 'Denmark')).geometry()

# Define map visual parameters
DW_CLASS = {
    'water' : '419bdf', # blue
    'trees' : '397d49', # green
    'grass' : '88b053', # light green
    'flooded_vegetation' : '7a87c6', # greyish blue
    'crops' : 'e49635', # orange
    'shrub_and_scrub' : 'dfc35a', # yellow
    'built' : 'c4281b', # red
    'bare' : 'a59b8f', # grey
    'snow_and_ice' : 'b39fe1', # purple
}

DW_VIZ_PARAMS = {
  'min' : 0,
  'max' : 8,
  'palette' : list(DW_CLASS.values()),
  'bands' : 'label'
}


In [3]:
# Calculate area of an image in km²
def dw_area(img):
    area_image = img.multiply(ee.Image.pixelArea())
    area = area_image.reduceRegion( 
        reducer = ee.Reducer.sum(),
        scale = 10,
        maxPixels = 1e13,
    )
    return ee.Number(area.get('label')).divide(pow(10,6))

# Calculate change matrix
# 9x9 matrix area classified per class vs area classified per class other year
def change_matrix(img, img_prev):
    for i in range(9):
        for j in range(9):
            dw_transition = img.eq(i).And(img_prev.eq(j))
            dw_transition_area = dw_area(dw_transition)
            display(dw_transition_area)


In [22]:
YEAR = 25

# DANISH AGRICULTURE AGENCY
dataset = ee.FeatureCollection("projects/" + PROJECT_ID + "/assets/Markblokke" + str(YEAR))
dataset_prev = ee.FeatureCollection("projects/" + PROJECT_ID + "/assets/Markblokke" + str(YEAR - 1))
# Transform data to raster format for lighter operations
daa_img = ee.Image.constant(0).paint(dataset, 1)
daa_img_prev = ee.Image.constant(0).paint(dataset_prev, 1)
# Image with are that changed from DAA (1 - new field, -1 - removed field)
daa_change_img = daa_img.subtract(daa_img_prev)
daa_change_img = daa_change_img.updateMask(daa_change_img.neq(0))


# DYNAMIC WORLD CLASSIFICATION OF DAA AREA
dw_img = ee.Image('projects/' + PROJECT_ID + '/assets/dw_20' + str(YEAR) + '_mode').clipToCollection(dataset)
dw_img_prev = ee.Image('projects/' + PROJECT_ID + '/assets/dw_20' + str(YEAR - 1) + '_mode').clipToCollection(dataset_prev)

## Change Matrix


In [6]:
change_matrix(dw_img, dw_img_prev)

#### New Fields Area

From the new field area, how did classification change? How was it classified before? How is it classified now?

New Field Area from 2024-2025: 90.54 km²

In [ ]:
change_matrix(dw_img.And(daa_change_img.eq(1)), dw_img_prev.And(daa_change_img.eq(1)))

#### Removed Fields Area

From the removed field area, how did classification change? How was it classified before? How is it classified now?

Removed Field Area from 2024-2025: 148.18 km²

In [ ]:
change_matrix(dw_img.And(daa_change_img.eq(-1)), dw_img_prev.And(daa_change_img.eq(-1)))

## Amount of Changes

In [34]:
dw_img_25 = ee.Image('projects/' + PROJECT_ID + '/assets/dw_2025_mode').clipToCollection(dataset)
dw_img_24 = ee.Image('projects/' + PROJECT_ID + '/assets/dw_2024_mode').clipToCollection(dataset)
dw_img_23 = ee.Image('projects/' + PROJECT_ID + '/assets/dw_2023_mode').clipToCollection(dataset)
dw_img_22 = ee.Image('projects/' + PROJECT_ID + '/assets/dw_2022_mode').clipToCollection(dataset)
dw_img_21 = ee.Image('projects/' + PROJECT_ID + '/assets/dw_2021_mode').clipToCollection(dataset)
dw_img_20 = ee.Image('projects/' + PROJECT_ID + '/assets/dw_2020_mode').clipToCollection(dataset)

dw_img_change_25_24 = dw_img_25.neq(dw_img_24)
dw_img_change_24_23 = dw_img_24.neq(dw_img_23)
dw_img_change_23_22 = dw_img_23.neq(dw_img_22)
dw_img_change_22_21 = dw_img_22.neq(dw_img_21)
dw_img_change_21_20 = dw_img_21.neq(dw_img_20)

change_collection = ee.ImageCollection.fromImages(
    [dw_img_change_25_24, dw_img_change_24_23, dw_img_change_23_22, dw_img_change_22_21, dw_img_change_21_20]
)

change_count_img = change_collection.sum()
area_img = ee.Image.pixelArea()
area_by_class = area_img.addBands(change_count_img)

stats = area_by_class.reduceRegion(
    reducer=ee.Reducer.sum().group(
        groupField=1,      # The index of the 'change_count_img' band
        groupName='class'  # The label for the dictionary keys
    ),
    geometry= DENMARK,
    scale=10,          
    maxPixels=1e13       
)

counts = change_count_img.reduceRegion(
    reducer=ee.Reducer.frequencyHistogram().unweighted(),
    geometry=DENMARK,
    scale=10,           # Set this to the native resolution of your image
    maxPixels=1e13       # Increase if working with large areas
)

results = stats.get('groups').getInfo()
print(counts.getInfo())

{'label': {'0': 315486081, '1': 55418995, '2': 79110098, '3': 37979090, '4': 12369488, '5': 1825984, 'null': 265202752}}


In [25]:
print(17739395326.638042 * 10 ** (-6))
print(3105677971.4612823  * 10 ** (-6))
print(4427591849.539228 * 10 ** (-6))
print(2122582646.3220909  * 10 ** (-6))
print(691735258.8870001 * 10 ** (-6))
print(102190748.02590838  * 10 ** (-6))

print(17739.39532663804 + 3105.677971461282 + 4427.591849539228 + 2122.5826463220906 + 691.7352588870001 + 102.19074802590838)

17739.39532663804
3105.677971461282
4427.591849539228
2122.5826463220906
691.7352588870001
102.19074802590838
28189.173800873552


In [29]:
dw_img_20_25 = ee.Image().addBands(dw_img_20)
dw_img_20_25 = dw_img_20_25.addBands(dw_img_21)
dw_img_20_25 = dw_img_20_25.addBands(dw_img_22)
dw_img_20_25 = dw_img_20_25.addBands(dw_img_23)
dw_img_20_25  = dw_img_20_25.addBands(dw_img_24)
dw_img_20_25 = dw_img_20_25.addBands(dw_img_25)

always_change = dw_img_20_25.updateMask(change_count_img.eq(5))



In [32]:
map = geemap.Map(scroll_wheel_zoom = False)
map.add_layer(
    always_change,
    DW_VIZ_PARAMS,
    'Dynamic World',
)
map

Map(center=[0, 0], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright', transp…

In [39]:
change_condition = dw_img_20_25.select('label').neq(dw_img_20_25.select('label_1')) \
    .And(dw_img_20_25.select('label').eq(dw_img_20_25.select('label_2')))

stats = change_condition.reduceRegion(
    reducer=ee.Reducer.sum().unweighted(),
    geometry=DENMARK,
    scale=10,
    maxPixels=1e13
)

pixel_count = stats.get('label') 
print('Changes from 2020 to 2021 that got reverted 2022:', pixel_count.getInfo())

change_condition = dw_img_20_25.select('label_1').neq(dw_img_20_25.select('label_2')) \
    .And(dw_img_20_25.select('label_1').eq(dw_img_20_25.select('label_3')))

stats = change_condition.reduceRegion(
    reducer=ee.Reducer.sum().unweighted(),
    geometry=DENMARK,
    scale=10,
    maxPixels=1e13
)

pixel_count = stats.get('label_1')
print('Changes from 2021 to 2022 that got reverted 2023:', pixel_count.getInfo())

change_condition = dw_img_20_25.select('label_2').neq(dw_img_20_25.select('label_3')) \
    .And(dw_img_20_25.select('label_2').eq(dw_img_20_25.select('label_4')))

stats = change_condition.reduceRegion(
    reducer=ee.Reducer.sum().unweighted(),
    geometry=DENMARK,
    scale=10,
    maxPixels=1e13
)

pixel_count = stats.get('label_2')
print('Changes from 2022 to 2023 that got reverted 2024:', pixel_count.getInfo())

change_condition = dw_img_20_25.select('label_3').neq(dw_img_20_25.select('label_4')) \
    .And(dw_img_20_25.select('label_3').eq(dw_img_20_25.select('label_5')))

stats = change_condition.reduceRegion(
    reducer=ee.Reducer.sum().unweighted(),
    geometry=DENMARK,
    scale=10,
    maxPixels=1e13
)

pixel_count = stats.get('label_3')
print('Changes from 2023 to 2024 that got reverted 2025:', pixel_count.getInfo())

Changes from 2020 to 2021 that got reverted 2022: 27990339
Changes from 2021 to 2022 that got reverted 2023: 47344349
Changes from 2022 to 2023 that got reverted 2024: 21580746
Changes from 2023 to 2024 that got reverted 2025: 23151792
